### Infer Author Nationality Without OpenAlex Country Data
When OpenAlex does not provide country information, the script infers nationality using:

- Country-related keywords  
- A predefined list of prominent universities and companies  
- City and state names
- Manual search

The non-determined records will be output to `04_not_determined.csv` for further inspection.

In [ ]:
import numpy as np
import pandas as pd
import yaml

In [ ]:
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

In [ ]:
import os
os.chdir('../../')

In [ ]:
import sys
sys.path.insert(1, 'modules/_utils/')

In [ ]:
from kw_matcher import country_name_match, top_uni_company_match, city_state_match

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
relevant_missing_geo = pd.read_parquet(dataset_config['path_processed'] + 'CN_CN/OA3_missing_geo.parquet')
relevant_missing_geo

In [ ]:
all_multi_country = pd.DataFrame()

In [ ]:
def post_process_results(data_to_search, all_multi_country, geo_result=None):
    if geo_result is None:
        new_data_to_search = data_to_search
        result_output = None
    else:
        geo_result = geo_result.dropna()
        print(f'Result: {len(geo_result)} total entries,')
        
        # Identify multi-country entries using the "!" marker
        multi_ctry_bool = geo_result.country.apply(lambda x: '!' in x)
        geo_result_multi = geo_result[multi_ctry_bool]
        geo_result_single = geo_result[~multi_ctry_bool]
        
        print(f'      * {len(geo_result_multi)} multi-country entries,')
        print(f'        {len(geo_result_single)} single-country entries,')
        
        # Append multi-country entries to the provided list (or DataFrame)
        all_multi_country = pd.concat([all_multi_country, geo_result_multi], ignore_index=True)
        
        # Merge single-country entries with data_to_search
        result_merged = data_to_search.merge(geo_result_single, on='affiliationame', how='left')
        result_output = result_merged[~result_merged.country.isna()]
        print(f'    {len(result_output)} found patent-paper-author entries, {len(result_output)/ len(data_to_search) * 100:.4f}%,')
        
        # Exclude multi-country affiliations from new_data_to_search
        multi_affiliations = all_multi_country['affiliationame'].unique()
        new_data_to_search = result_merged[
            result_merged.country.isna() & ~result_merged.affiliationame.isin(multi_affiliations)
        ].drop(columns=['country'])
        print(f'    {len(new_data_to_search)} remaining patent-paper-author entries,')
        
    new_geo_db = pd.DataFrame(new_data_to_search.affiliationame.unique(), columns=['affiliationame'])
    print(f'{len(new_geo_db)} remaining to search affiliationames.')
    
    return result_output, new_data_to_search, new_geo_db, all_multi_country


In [ ]:
_, data_to_search1, geo_db1, _ = post_process_results(relevant_missing_geo, all_multi_country)

### (1) Identify affiliation by explicit country names

In [ ]:
geo_db1['country'] = geo_db1.affiliationame.parallel_map(country_name_match)

In [ ]:
result_output1, data_to_search2, geo_db2, all_multi_country = post_process_results(data_to_search1, all_multi_country, geo_db1)

In [ ]:
result_output1

In [ ]:
result_output1.to_parquet(dataset_config['path_processed'] + 'CN_CN/POST1_recognized_by_country_name.parquet')

### (2) Identify affiliation by well-known universities and firms

The university list is based on the Top 100 QS World University Rankings (Website: https://www.topuniversities.com/world-university-rankings).

The company list is based on the Fortune 500 (Website: https://fortune.com/ranking/global500/search/).

In [ ]:
geo_db2['country'] = geo_db2.affiliationame.parallel_map(top_uni_company_match)

In [ ]:
result_output2, data_to_search3, geo_db3, all_multi_country = post_process_results(data_to_search2, all_multi_country, geo_db2)

In [ ]:
result_output2

In [ ]:
result_output2.to_parquet(dataset_config['path_processed'] + 'CN_CN/POST1_recognized_by_top_uni_company.parquet')

### (3) Identify affiliation by manual supplementation

In [ ]:
geo_db3['country'] = geo_db3.affiliationame.parallel_map(city_state_match)

In [ ]:
result_output3, data_to_search4, geo_db4, all_multi_country = post_process_results(data_to_search3, all_multi_country, geo_db3)

In [ ]:
result_output3

In [ ]:
result_output3.to_parquet(dataset_config['path_processed'] + 'CN_CN/POST1_recognized_by_city_state_match.parquet')

In [ ]:
data_to_search4.to_parquet(dataset_config['path_processed'] + 'CN_CN/POST1_not_recognized.parquet')